<a href="https://colab.research.google.com/github/divijdawar/CUDA/blob/main/CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile gemm_naive.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

// CUDA kernel for naive GEMM
__global__ void gemm_naive(int M, int N, int K, float alpha, const float *A, const float *B, float beta, float *C) {
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    const int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x < M && y < N) {
        float tmp = 0.0f;
        for (int i = 0; i < K; ++i) {
            tmp += A[x * K + i] * B[i * N + y];
        }
        // C = alpha * (A @ B) + beta * C
        C[x * N + y] = alpha * tmp + beta * C[x * N + y];
    }
}

int main() {

    const int M = 4092;
    const int K = 4092;
    const int N = 4092;

    // Host (CPU) memory allocation
    float *h_A = (float*)malloc(M * K * sizeof(float));
    float *h_B = (float*)malloc(K * N * sizeof(float));
    float *h_C = (float*)malloc(M * N * sizeof(float));

    // Initialize matrices with random values
    for (int i = 0; i < M * K; i++) h_A[i] = (float)(rand() % 100) / 100.0;
    for (int i = 0; i < K * N; i++) h_B[i] = (float)(rand() % 100) / 100.0;
    for (int i = 0; i < M * N; i++) h_C[i] = 0.0f;  // Initialize C to zeros

    // Device (GPU) memory allocation
    float *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, M * K * sizeof(float));
    cudaMalloc((void**)&d_B, K * N * sizeof(float));
    cudaMalloc((void**)&d_C, M * N * sizeof(float));

    // Copy data from host to device
    cudaMemcpy(d_A, h_A, M * K * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, K * N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_C, h_C, M * N * sizeof(float), cudaMemcpyHostToDevice);

    // Define grid and block sizes
    dim3 blockDim(32, 32, 1);
    dim3 gridDim(CEIL_DIV(M, 32), CEIL_DIV(N, 32), 1);

    // Create CUDA events for timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Launch kernel and measure execution time
    cudaEventRecord(start);
    gemm_naive<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    // Compute elapsed time
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Kernel Execution Time: %f ms\n", milliseconds);

    // Measure memory transfer time (GPU → CPU)
    cudaEvent_t startMem, stopMem;
    cudaEventCreate(&startMem);
    cudaEventCreate(&stopMem);

    cudaEventRecord(startMem);
    cudaMemcpy(h_C, d_C, M * N * sizeof(float), cudaMemcpyDeviceToHost);
    cudaEventRecord(stopMem);
    cudaEventSynchronize(stopMem);

    float memTransferTime;
    cudaEventElapsedTime(&memTransferTime, startMem, stopMem);
    printf("Memory Transfer Time (Device to Host): %f ms\n", memTransferTime);

    // Cleanup: Free device and host memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing gemm_naive.cu


In [2]:
!nvcc gemm_naive.cu -o gemm_naive

In [3]:
!./gemm_naive

Kernel Execution Time: 51.890369 ms
Memory Transfer Time (Device to Host): 15.000576 ms


In [4]:
%%writefile  sgemm_coalescing.cu
/**
A T4 GPU offers 8.1 TFLOPS of FP 32 performance and 320 GB/s of memory bandwith.
2 * 4092 * 4092 * 4092 = 137.2 GFLOPs
So 8.1 TFLOPS / 137.2 GFLOPs = 16.ms
The last exeution was at 54.33 ms, so we have lots of improvement to make.
**/

#include <stdio.h>
#include <cuda_runtime.h>
#define BLOCKSIZE 32
#define CEIL_DIV(M, N) (((M) + (N) - 1) / (N))

__global__ void sgemm_coalescing(int M, int N, int K, float alpha, const float *A, const float *B, float beta, float *C){
  const int x = blockIdx.x * BLOCKSIZE + (threadIdx.x / BLOCKSIZE); // Blocksize defines the number of threads in a block
  const int y = blockIdx.y * BLOCKSIZE + (threadIdx.y % BLOCKSIZE);

  if (x < M && y < N){
    float tmp = 0.0;
    for (int i = 0; i < K; ++i){
      tmp += A[x * K + i] * B[i * N + y]; // Matrix A is M x K, Matrix B is K x N
    }
    C[x * N + y] = alpha * tmp + beta * C[x * N + y];
  }
}

int main(){
    const int M = 4092;
    const int K = 4092;
    const int N = 4092;

    // Host (CPU) memory allocation
    float *h_A = (float*)malloc(M * K * sizeof(float));
    float *h_B = (float*)malloc(K * N * sizeof(float));
    float *h_C = (float*)malloc(M * N * sizeof(float));

    // Initialize matrices with random values
    for (int i = 0; i < M * K; i++) h_A[i] = (float)(rand() % 100) / 100.0;
    for (int i = 0; i < K * N; i++) h_B[i] = (float)(rand() % 100) / 100.0;
    for (int i = 0; i < M * N; i++) h_C[i] = 0.0f;  // Initialize C to zeros

    // Device (GPU) memory allocation
    float *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, M * K * sizeof(float));
    cudaMalloc((void**)&d_B, K * N * sizeof(float));
    cudaMalloc((void**)&d_C, M * N * sizeof(float));

    // Copy data from host to device
    cudaMemcpy(d_A, h_A, M * K * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, K * N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_C, h_C, M * N * sizeof(float), cudaMemcpyHostToDevice);

    // Define grid and block sizes
    dim3 blockDim(32*32);
    dim3 gridDim(CEIL_DIV(M, 32), CEIL_DIV(N, 32), 1);

    // Create CUDA events for timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Launch kernel and measure execution time
    cudaEventRecord(start);
    sgemm_coalescing<<<gridDim, blockDim>>>(M, N, K, 1.0f, d_A, d_B, 0.0f, d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    // Compute elapsed time
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Kernel Execution Time: %f ms\n", milliseconds);

    // Compute GFLOPs
    float gflops = 2.0 * M * N * K / (milliseconds * 1e3);
    printf("GFLOPs: %f\n", gflops);

    // Measure memory transfer time (GPU → CPU)
    cudaEvent_t startMem, stopMem;
    cudaEventCreate(&startMem);
    cudaEventCreate(&stopMem);

    cudaEventRecord(startMem);
    cudaMemcpy(h_C, d_C, M * N * sizeof(float), cudaMemcpyDeviceToHost);
    cudaEventRecord(stopMem);
    cudaEventSynchronize(stopMem);

    float memTransferTime;
    cudaEventElapsedTime(&memTransferTime, startMem, stopMem);
    printf("Memory Transfer Time (Device to Host): %f ms\n", memTransferTime);

    // Cleanup: Free device and host memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}


Writing sgemm_coalescing.cu


In [5]:
!nvcc sgemm_coalescing.cu -o sgemm_coalescing

In [6]:
!./sgemm_coalescing

Kernel Execution Time: 11.571872 ms
GFLOPs: 11842224.000000
Memory Transfer Time (Device to Host): 16.499071 ms


In [11]:
%%writefile matrixMulOptimized.cu
#include <cuda_runtime.h>
#include <stdio.h>

// Define block and thread dimensions
#define BM 128 // Block tile size for M dimension
#define BN 128 // Block tile size for N dimension
#define BK 8   // Block tile size for K dimension
#define TM 8   // Thread tile size (per thread)

// CUDA kernel for matrix multiplication with thread-local caching
__global__ void matrixMulOptimized(float* C, float* A, float* B, int M, int N, int K) {
    // Block indices
    int blockRow = blockIdx.y;
    int blockCol = blockIdx.x;

    // Thread indices
    int threadRow = threadIdx.y;
    int threadCol = threadIdx.x;

    // Starting position for this block
    int startM = blockRow * BM;
    int startN = blockCol * BN;

    // Shared memory allocation
    __shared__ float As[BM * BK];
    __shared__ float Bs[BK * BN];

    // Index calculations for loading into shared memory
    int innerRowA = threadIdx.y;
    int innerColA = threadIdx.x;
    int innerRowB = threadIdx.y;
    int innerColB = threadIdx.x;

    // Adjustments for non-perfect grid dimensions
    if (startM + innerRowA >= M || blockCol * BK + innerColA >= K) {
        As[innerRowA * BK + innerColA] = 0.0f;
    } else {
        As[innerRowA * BK + innerColA] = A[(startM + innerRowA) * K + blockCol * BK + innerColA];
    }

    if (blockRow * BK + innerRowB >= K || startN + innerColB >= N) {
        Bs[innerRowB * BN + innerColB] = 0.0f;
    } else {
        Bs[innerRowB * BN + innerColB] = B[(blockRow * BK + innerRowB) * N + startN + innerColB];
    }

    // Point to this block's section of A and B
    A = A + startM * K;
    B = B + startN;

    // allocate thread-local cache for results in registerfile
    float threadResults[TM] = {0.0};

    // outer loop over block tiles
    for (uint bkIdx = 0; bkIdx < K; bkIdx += BK) {
      // populate the SMEM caches (same as before)
      if (startM + innerRowA < M && bkIdx + innerColA < K) {
          As[innerRowA * BK + innerColA] = A[innerRowA * K + innerColA];
      } else {
          As[innerRowA * BK + innerColA] = 0.0f;
      }

      if (bkIdx + innerRowB < K && startN + innerColB < N) {
          Bs[innerRowB * BN + innerColB] = B[innerRowB * N + innerColB];
      } else {
          Bs[innerRowB * BN + innerColB] = 0.0f;
      }

      __syncthreads();

      // advance blocktile for outer loop
      A += BK;
      B += BK * N;

      // calculate per-thread results
      for (uint dotIdx = 0; dotIdx < BK; ++dotIdx) {
        // we make the dotproduct loop the outside loop, which facilitates
        // reuse of the Bs entry, which we can cache in a tmp var.
        float Btmp = Bs[dotIdx * BN + threadCol];
        for (uint resIdx = 0; resIdx < TM; ++resIdx) {
          threadResults[resIdx] +=
              As[(threadRow * TM + resIdx) * BK + dotIdx] * Btmp;
        }
      }
      __syncthreads();
    }

    // Write results to global memory
    for (uint resIdx = 0; resIdx < TM; ++resIdx) {
        int outputRow = startM + threadRow * TM + resIdx;
        int outputCol = startN + threadCol;

        if (outputRow < M && outputCol < N) {
            C[outputRow * N + outputCol] = threadResults[resIdx];
        }
    }
}

// Utility function to initialize a matrix
void initMatrix(float* matrix, int rows, int cols) {
    for (int i = 0; i < rows * cols; i++) {
        matrix[i] = (float)(rand() % 10) / 10.0f;
    }
}

// Utility function to verify results against CPU computation
bool verifyResults(float* C, float* A, float* B, int M, int N, int K) {
    float* cpuC = (float*)malloc(M * N * sizeof(float));
    memset(cpuC, 0, M * N * sizeof(float));

    // Compute C = A * B on CPU
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < K; k++) {
                sum += A[i * K + k] * B[k * N + j];
            }
            cpuC[i * N + j] = sum;
        }
    }

    // Compare GPU results with CPU results
    bool correct = true;
    for (int i = 0; i < M * N; i++) {
        // Use small epsilon for floating point comparison
        if (fabs(C[i] - cpuC[i]) > 1e-4) {
            printf("Mismatch at index %d: GPU=%f, CPU=%f\n", i, C[i], cpuC[i]);
            correct = false;
            break;
        }
    }

    free(cpuC);
    return correct;
}

int main() {

    // Define matrix dimensions
    int M = 1024;  // Rows of A and C
    int N = 1024;  // Columns of B and C
    int K = 1024;  // Columns of A and rows of B

    printf("Matrix dimensions: A(%d x %d), B(%d x %d), C(%d x %d)\n", M, K, K, N, M, N);

    size_t sizeA = M * K * sizeof(float);
    size_t sizeB = K * N * sizeof(float);
    size_t sizeC = M * N * sizeof(float);

    printf("Memory requirements: A: %.2f MB, B: %.2f MB, C: %.2f MB, Total: %.2f MB\n",
           sizeA / (1024.0f * 1024.0f),
           sizeB / (1024.0f * 1024.0f),
           sizeC / (1024.0f * 1024.0f),
           (sizeA + sizeB + sizeC) / (1024.0f * 1024.0f));

    // Allocate host memory
    float* h_A = (float*)malloc(sizeA);
    float* h_B = (float*)malloc(sizeB);
    float* h_C = (float*)malloc(sizeC);

    // Initialize matrices
    printf("Initializing matrices...\n");
    initMatrix(h_A, M, K);
    initMatrix(h_B, K, N);

    // Allocate device memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, sizeA);
    cudaMalloc(&d_B, sizeB);
    cudaMalloc(&d_C, sizeC);

    // Create CUDA events for timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float milliseconds = 0;

    // Measure host to device transfer time
    cudaEventRecord(start);
    cudaMemcpy(d_A, h_A, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, sizeB, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Host to device memory copy time: %.2f ms\n", milliseconds);
    printf("Host to device bandwidth: %.2f GB/s\n",
           (sizeA + sizeB) / (milliseconds * 1.0e6));

    // Calculate grid and block dimensions
    dim3 blockDim(BN / TM, BM / TM);
    dim3 gridDim((N + BN - 1) / BN, (M + BM - 1) / BM);

    // Launch kernel with timing
    printf("Launching kernel with grid(%d,%d) and block(%d,%d)\n",
           gridDim.x, gridDim.y, blockDim.x, blockDim.y);
    printf("Thread tile size (TM): %d\n", TM);
    printf("Block dimensions: BM=%d, BN=%d, BK=%d\n", BM, BN, BK);

    // Warm-up run (not timed)
    matrixMulOptimized<<<gridDim, blockDim>>>(d_C, d_A, d_B, M, N, K);
    cudaDeviceSynchronize();

    // Timed kernel execution
    cudaEventRecord(start);
    matrixMulOptimized<<<gridDim, blockDim>>>(d_C, d_A, d_B, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milliseconds, start, stop);

    // Calculate performance metrics
    double flops = 2.0 * M * N * K; // multiply-add counts as 2 operations
    double teraFlops = (flops / (milliseconds / 1000.0)) / 1e12;

    printf("\n--- Performance Results ---\n");
    printf("Kernel execution time: %.2f ms\n", milliseconds);
    printf("Performance: %.2f TFLOPS\n", teraFlops);
    printf("Arithmetic Intensity: %.1f FLOP/byte\n",
           flops / ((sizeA + sizeB + sizeC)));

    // Calculate theoretical peak performance
    int deviceId;
    cudaGetDevice(&deviceId);
    cudaDeviceProp deviceProp;
    cudaGetDeviceProperties(&deviceProp, deviceId);

    float clockRateGHz = deviceProp.clockRate / 1e6;
    int smCount = deviceProp.multiProcessorCount;

    // This is an estimate that works reasonably well for recent NVIDIA GPUs
    // Adjust based on specific architecture if needed
    int fpUnitsPerSM = 64; // Typical for recent architectures

    double theoreticalPeakTFLOPS = 2.0 * clockRateGHz * smCount * fpUnitsPerSM / 1000.0;
    printf("Theoretical peak performance: ~%.2f TFLOPS\n", theoreticalPeakTFLOPS);
    printf("Achieved: %.1f%% of peak performance\n",
           (teraFlops / theoreticalPeakTFLOPS) * 100.0);

    // Measure device to host transfer time
    cudaEventRecord(start);
    cudaMemcpy(h_C, d_C, sizeC, cudaMemcpyDeviceToHost);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Device to host memory copy time: %.2f ms\n", milliseconds);
    printf("Device to host bandwidth: %.2f GB/s\n",
           sizeC / (milliseconds * 1.0e6));

    // Check for errors
    cudaError_t cudaError = cudaGetLastError();
    if (cudaError != cudaSuccess) {
        printf("CUDA error: %s\n", cudaGetErrorString(cudaError));
        return -1;
    }

    // Verify results
    printf("\nVerifying results...\n");
    bool correct = verifyResults(h_C, h_A, h_B, M, N, K);
    if (correct) {
        printf("Results match!\n");
    } else {
        printf("Results do not match!\n");
    }

    // Free memory
    free(h_A);
    free(h_B);
    free(h_C);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Overwriting matrixMulOptimized.cu


In [12]:
!nvcc matrixMulOptimized.cu -o matrixMulOptimized

In [14]:
!./matrixMulOptimized

Matrix dimensions: A(1024 x 1024), B(1024 x 1024), C(1024 x 1024)
Memory requirements: A: 4.00 MB, B: 4.00 MB, C: 4.00 MB, Total: 12.00 MB
Initializing matrices...
Host to device memory copy time: 2.05 ms
Host to device bandwidth: 4.10 GB/s
Launching kernel with grid(8,8) and block(16,16)
Thread tile size (TM): 8
Block dimensions: BM=128, BN=128, BK=8

--- Performance Results ---
Kernel execution time: 0.01 ms
Performance: 411.71 TFLOPS
Arithmetic Intensity: 170.7 FLOP/byte
Theoretical peak performance: ~8.14 TFLOPS
Achieved: 5057.4% of peak performance
Device to host memory copy time: 3.55 ms
Device to host bandwidth: 1.18 GB/s
CUDA error: the provided PTX was compiled with an unsupported toolchain.
